In [2]:
# 가상환경 실행 : .\.venv\Scripts\activate.ps1

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma
#from langchain_community.llms import Ollama
#from langchain_community.chat_models import ChatOllama
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain.agents import initialize_agent, AgentType
#from langchain_ollama import OllamaEmbeddings
import os
from langgraph.prebuilt import ToolNode
from typing import Literal
#from langgraph.graph import END
from langgraph.graph import START, END
from langgraph.graph import MessagesState, StateGraph
#from langchain_core.prompts import PromptTemplate
from pathlib import Path
from langchain.prompts import PromptTemplate


In [3]:
# model cell

embeddings = OllamaEmbeddings(
    model="bge-m3"
)

llm = ChatOllama(
    model="qwen3:4b"
)

C:\Users\worb1\AppData\Local\Temp\ipykernel_12500\3511307189.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


In [6]:

p_path = './chroma_db'
print("[DEBUG] cwd       =", Path.cwd())
if os.path.exists(p_path) and len(os.listdir(p_path)) > 0:
    #기존 벡터 DB 가 존재할 경우.
    print("Vector DB 존재. 불러오기 시작.")
    collection_name = 'requirements_list'

    persist_directory = p_path
    
    vector_store = Chroma(
        embedding_function=embeddings,
        collection_name = collection_name,
        persist_directory = persist_directory
    )
    print("Vector DB 불러오기 완료.")
    
else:
    print("Vector DB 부재. 생성 시작.")
    # 1. 문서 로드
    base_dir = "."
    file_path = base_dir + "/docs" + "/RFP_requirements.md"

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    

    doc_parse_prompt = PromptTemplate.from_template(r"""
        당신은 SI 프로젝트 요구사항 정의서를 읽고, 개발자가 바로 사용할 수 있는 JSON 체크리스트로 변환하는 도우미입니다.
        입력으로 SFR 섹션 하나(마크다운)가 주어집니다.

        [출력 형식 규칙]
        1) 출력은 오직 JSON만. 앞/뒤 설명, 마크다운, 코드펜스 금지.
        2) 스키마는 아래와 동일해야 함:
        {{
        "요구사항ID": "<SFR-XXX>",
        "기능명": "<한 줄 요약>",
        "구현항목": [
            {{
            "하위ID": "<SFR-XXX-01>",
            "내용": "<구현해야 할 기능>",
            "구현시참고사항": "<개발 시 유의/맥락 1문장>"
            }}
        ]
        }}
        3) "하위ID"는 요구사항ID에서 파생: <SFR-XXX-01>, <SFR-XXX-02> … 두 자리 증가.
        4) "내용"은 입력의 '소분류'와 그 하위 불릿들을 분석해, 실행 가능 문장으로 간결(최대 30자)하게. 핵심 동사를 앞에 둔다.
        - 예: "엑셀 업로드 통한 대량 과정 등록", "과정 리스트 조회 및 수료 처리"
        5) "구현시참고사항"은 의도/운영 관점에서 1문장(최대 40자)으로 요약.
        - 정책, 예외, 대량처리, 변경반영, 추적성 등의 키워드를 적절히 반영하되 사실 확장/추측 금지.
        6) 근거문서/비고/메타 정보는 JSON에 포함하지 않는다.
        7) 입력에 없는 기능은 생성하지 않는다(할루시네이션 금지). 한글만 사용하고, 따옴표는 ASCII(")만 사용.

        [입력 섹션]

        {section}

        위 섹션을 단일 JSON으로 변환하시오.
                                                    
        모든 출력은 <output> 태그 안에 담아서 추출하기 좋게 정리해주세요.
        출력 예시 : 
        <outout>
            당신이 생각한 모든 출력
        </output>
    """)

    msg = doc_parse_prompt.format(section=content)
    response = llm.invoke(msg)


    import re
    import json
    from langchain.schema import Document
    def extract_output_after_removing_think(text: str, return_json=False):
        no_think = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)

        blocks = re.findall(r"<output>.*?</output>", no_think, flags=re.DOTALL | re.IGNORECASE)

        picked = None
        for b in reversed(blocks):
            if re.search(r"<output>\s*\S", b, flags=re.DOTALL | re.IGNORECASE):
                picked = b
                break

        if picked is None:
            return None

        if return_json:
            inner = re.search(r"<output>(.*)</output>", picked, flags=re.DOTALL | re.IGNORECASE).group(1).strip()
            return json.loads(inner) 
        else:
            return picked

    text = response.content 
    only_output_block = extract_output_after_removing_think(text) 
    parsed_json = extract_output_after_removing_think(text, return_json=True)

    vector_db_items = []
    for idx1, sfr in enumerate(parsed_json):
        for idx2, detail_sfr in enumerate(sfr['구현항목']):
            req_id = parsed_json[idx1]['요구사항ID']
            sub_item_id = detail_sfr['하위ID']
            str_detail_sft = json.dumps(detail_sfr)
            # print(req_id)
            # print(sub_item_id)
            # print(detail_sfr)
            doc = Document(page_content=str_detail_sft,metadata={"source": req_id, "list_name": sub_item_id, "idx": idx2})
            vector_db_items.append(doc)

    # 3. 벡터 스토어 생성
    persist_directory = "./fastapi-client/chroma_db"
    collection_name = 'requirements_list'
    vector_store = Chroma.from_documents(
        documents=vector_db_items, 
        embedding=embeddings, 
        persist_directory=persist_directory,
        collection_name=collection_name    
    )
    print("Vector DB 생성 완료.")

[DEBUG] cwd       = c:\Users\worb1\Documents\vscode-project\ppm\fastapi-client
Vector DB 부재. 생성 시작.


JSONDecodeError: Extra data: line 17 column 1 (char 332)

In [7]:
print("Vector DB 부재. 생성 시작.")
# 1. 문서 로드
base_dir = "."
file_path = base_dir + "/docs" + "/RFP_requirements.md"

with open(file_path, "r", encoding="utf-8") as f:
    content = f.read()



doc_parse_prompt = PromptTemplate.from_template(r"""
    당신은 SI 프로젝트 요구사항 정의서를 읽고, 개발자가 바로 사용할 수 있는 JSON 체크리스트로 변환하는 도우미입니다.
    입력으로 SFR 섹션 하나(마크다운)가 주어집니다.

    [출력 형식 규칙]
    1) 출력은 오직 JSON만. 앞/뒤 설명, 마크다운, 코드펜스 금지.
    2) 스키마는 아래와 동일해야 함:
    {{
    "요구사항ID": "<SFR-XXX>",
    "기능명": "<한 줄 요약>",
    "구현항목": [
        {{
        "하위ID": "<SFR-XXX-01>",
        "내용": "<구현해야 할 기능>",
        "구현시참고사항": "<개발 시 유의/맥락 1문장>"
        }}
    ]
    }}
    3) "하위ID"는 요구사항ID에서 파생: <SFR-XXX-01>, <SFR-XXX-02> … 두 자리 증가.
    4) "내용"은 입력의 '소분류'와 그 하위 불릿들을 분석해, 실행 가능 문장으로 간결(최대 30자)하게. 핵심 동사를 앞에 둔다.
    - 예: "엑셀 업로드 통한 대량 과정 등록", "과정 리스트 조회 및 수료 처리"
    5) "구현시참고사항"은 의도/운영 관점에서 1문장(최대 40자)으로 요약.
    - 정책, 예외, 대량처리, 변경반영, 추적성 등의 키워드를 적절히 반영하되 사실 확장/추측 금지.
    6) 근거문서/비고/메타 정보는 JSON에 포함하지 않는다.
    7) 입력에 없는 기능은 생성하지 않는다(할루시네이션 금지). 한글만 사용하고, 따옴표는 ASCII(")만 사용.

    [입력 섹션]

    {section}

    위 섹션을 단일 JSON으로 변환하시오.
                                                
    모든 출력은 <output> 태그 안에 담아서 추출하기 좋게 정리해주세요.
    출력 예시 : 
    <outout>
        당신이 생각한 모든 출력
    </output>
""")

msg = doc_parse_prompt.format(section=content)
response = llm.invoke(msg)


Vector DB 부재. 생성 시작.


In [10]:
response.content

'<think>\nOkay, let\'s tackle this. The user wants me to convert the given SFR sections into a JSON checklist for developers. First, I need to look at each SFR section provided and extract the relevant information.\n\nStarting with SFR-099: Test Logic Implementation. The main requirement is to have a function that prints "호출 됨" and "1등하자" when an API is called and returns "hello world". The sub-items here are just one, so I\'ll map that into the JSON structure. The key points are the print statements and the return value.\n\nNext, SFR-004: Program and Course Setup. The main function here is about setting up and managing courses, including Excel uploads, course list retrieval, and managing enrollment and changes. There are several sub-items here. I need to break each of these into separate entries under the main requirement. For example, the Excel upload, course list retrieval, enrollment management, etc. Each of these should be a separate item in the "구현항목" array with their own "하위ID" 

In [27]:
import re
import json
from langchain.schema import Document
def extract_output_after_removing_think(text: str, return_json=False):
    no_think = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)

    blocks = re.findall(r"<output>.*?</output>", no_think, flags=re.DOTALL | re.IGNORECASE)

    picked = None
    for b in reversed(blocks):
        if re.search(r"<output>\s*\S", b, flags=re.DOTALL | re.IGNORECASE):
            picked = b
            break

    if picked is None:
        return None

    if return_json:
        inner = re.search(r"<output>(.*)</output>", picked, flags=re.DOTALL | re.IGNORECASE).group(1).strip()
        return json.loads(inner) 
    else:
        return picked

text = response.content 
tmp_parsed_json = extract_output_after_removing_think(text) 

# 1) <output> 태그 제거
cleaned = re.sub(r"</?output>", "", tmp_parsed_json, flags=re.DOTALL).strip()

# 2) JSON 여러 개를 리스트로 감싸기
json_str = "[" + cleaned + "]"

json_str

# 중간에 , 삽입
fixed = json_str.replace('}\n{', '},\n{')

# 문자열 → 리스트 변환
parsed_json = json.loads(fixed)

In [29]:
vector_db_items = []
for idx1, sfr in enumerate(parsed_json):
    for idx2, detail_sfr in enumerate(sfr['구현항목']):
        req_id = parsed_json[idx1]['요구사항ID']
        sub_item_id = detail_sfr['하위ID']
        str_detail_sft = json.dumps(detail_sfr)
        # print(req_id)
        # print(sub_item_id)
        # print(detail_sfr)
        doc = Document(page_content=str_detail_sft,metadata={"source": req_id, "list_name": sub_item_id, "idx": idx2})
        vector_db_items.append(doc)

# 3. 벡터 스토어 생성
persist_directory = "./chroma_db"
collection_name = 'requirements_list'
vector_store = Chroma.from_documents(
    documents=vector_db_items, 
    embedding=embeddings, 
    persist_directory=persist_directory,
    collection_name=collection_name    
)
print("Vector DB 생성 완료.")


Vector DB 생성 완료.


In [30]:
retriever = vector_store.as_retriever(
    search_kwargs={'k': 1}
)

In [31]:
answer = retriever.invoke("1등하자 를 출력하는 로그를 요구하는 사항은?")
# print(answer[0].metadata['source'])
# print(answer[0].metadata['list_name'])
# data = json.loads(answer[0].page_content)
# print(json.dumps(data, indent=2, ensure_ascii=False))
answer

[Document(id='242363f4-184b-40bd-8c2e-8d32ef89a325', metadata={'list_name': 'SFR-099-02', 'idx': 1, 'source': 'SFR-099'}, page_content='{"\\ud558\\uc704ID": "SFR-099-02", "\\ub0b4\\uc6a9": "api \\ud638\\ucd9c \\uc2dc \'1\\ub4f1\\ud558\\uc790\' \\ucd9c\\ub825", "\\uad6c\\ud604\\uc2dc\\ucc38\\uace0\\uc0ac\\ud56d": "\\ud638\\ucd9c \\uc2dc\\uc2a4\\ud15c \\uba54\\uc2dc\\uc9c0 \\uc815\\ucc45 \\ubc18\\uc601"}')]

In [32]:
retriever.search_kwargs["filter"] = {"source": "SFR-011"}
answer = retriever.invoke("1등하자 를 출력하는 로그를 요구하는 사항은?")
print(answer)
# print(answer[0].metadata['source'])
# print(answer[0].metadata['list_name'])
# data = json.loads(answer[0].page_content)
# print(json.dumps(data, indent=2, ensure_ascii=False))

[]


In [33]:
retriever.search_kwargs["filter"] = {"source": "SFR-099"}
answer = retriever.invoke("1등하자 를 출력하는 로그를 요구하는 사항은?")
print(answer)
# print(answer[0].metadata['source'])
# print(answer[0].metadata['list_name'])
# data = json.loads(answer[0].page_content)
# print(json.dumps(data, indent=2, ensure_ascii=False))

[Document(id='242363f4-184b-40bd-8c2e-8d32ef89a325', metadata={'source': 'SFR-099', 'idx': 1, 'list_name': 'SFR-099-02'}, page_content='{"\\ud558\\uc704ID": "SFR-099-02", "\\ub0b4\\uc6a9": "api \\ud638\\ucd9c \\uc2dc \'1\\ub4f1\\ud558\\uc790\' \\ucd9c\\ub825", "\\uad6c\\ud604\\uc2dc\\ucc38\\uace0\\uc0ac\\ud56d": "\\ud638\\ucd9c \\uc2dc\\uc2a4\\ud15c \\uba54\\uc2dc\\uc9c0 \\uc815\\ucc45 \\ubc18\\uc601"}')]


[Document(id='10f379c9-ae55-4910-adce-9da6958d5118', metadata={'source': './fastapi-client/docs/RFP_requirements.md'}, page_content='- **요구사항명**: 교육 결과 관리\n- **설명**:  \n  교육별 결과보고서와 실시 현황을 체계적으로 관리하고, 교육 이수에 실패한 학습자에 대해 자동화된 알림을 통해 수강 독려가 가능해야 한다.\n\n- **근거문서**:\n  - RFP III‑3 “상세 요구사항” SFR‑010 :contentReference[oaicite:6]{index=6}\n\n- **비고**:\n  - 산출물: 요구사항정의서, 요구사항추적표, 화면설계서  \n\n## SFR-011: 수강신청 정산관리 기능')]

[]
